# Lab 1.2: Reason Codes from a Scorecard

Run the cells from top to bottom. You do not need to change any code. Each checkpoint tells you what to look for.

Data: German Credit (Statlog), 1,000 applicants, downloaded from OpenML. See `data/german-credit.md` for the data card and license.

## 1. Setup and data

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def load_credit():
    """German Credit (Statlog) from OpenML; falls back to data/german_credit.csv if offline."""
    try:
        from sklearn.datasets import fetch_openml
        d = fetch_openml("credit-g", version=1, as_frame=True)
        df = d.frame.copy()
    except Exception:
        df = pd.read_csv("../../../data/german_credit.csv")
    df["bad"] = (df["class"] == "bad").astype(int)
    return df.drop(columns=["class"])


## 2. Turn each feature into bands

In [ ]:
FEATURES = ["checking_status", "duration", "credit_history", "credit_amount",
            "savings_status", "employment", "age"]

def bands(df):
    out = pd.DataFrame(index=df.index)
    for f in FEATURES:
        col = df[f]
        if pd.api.types.is_numeric_dtype(col):
            out[f] = pd.qcut(col, 4, duplicates="drop").astype(str)
        else:
            out[f] = col.astype(str)
    return out


## 3. Fit a points-based scorecard and compute reasons

In [ ]:
def fit_scorecard(df, pdo=20, base=600):
    B = bands(df)
    X = pd.get_dummies(B, prefix_sep="=")
    y = df["bad"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    lr = LogisticRegression(max_iter=2000, C=1.0).fit(Xtr, ytr)
    factor = pdo / np.log(2)
    # higher points = lower risk; points per band = -coef * factor
    pts = pd.Series(-lr.coef_[0] * factor, index=X.columns)
    table = []
    for f in FEATURES:
        cols = [c for c in X.columns if c.startswith(f + "=")]
        p = pts[cols] - pts[cols].min()          # worst band gets 0 points
        for c in cols:
            table.append((f, c.split("=", 1)[1], round(float(p[c]))))
    card = pd.DataFrame(table, columns=["feature", "band", "points"])
    return card, B, Xte.index

def score(card, B):
    look = {(r.feature, r.band): r.points for r in card.itertuples()}
    P = pd.DataFrame({f: [look[(f, b)] for b in B[f]] for f in FEATURES}, index=B.index)
    return P

def reasons(card, P, k=2):
    mx = card.groupby("feature")["points"].max()
    gap = mx[FEATURES].values - P[FEATURES].values
    order = np.argsort(-gap, axis=1)[:, :k]
    return [[(FEATURES[j], int(gap[i, j])) for j in order[i]] for i in range(len(P))]


## 4. Run it

**Checkpoint 1:** the scorecard table shows, for each feature, the points for each band. The worst band of every feature has 0 points.

In [ ]:
df = load_credit()
card, B, test_idx = fit_scorecard(df)
card


**Checkpoint 2:** the five applicants below are in the lowest 30 percent of total points on the test set. That cutoff is invented for the lab; real lenders set their own.

In [ ]:
P = score(card, B.loc[test_idx])
total = P.sum(axis=1)
cutoff = total.quantile(0.3)
refused = P[total < cutoff].head(5)
for idx, rs in zip(refused.index, reasons(card, refused)):
    print(f"applicant {idx}: {int(total[idx])} points; reasons: " + "; ".join(f"{f} ({g} below max)" for f, g in rs))


## 5. Your write-up

For each of the five applicants, copy the two reasons. Then write two sentences: what do these reasons not tell the applicant? (Hint: do they say what to change, and could the applicant change it?)